# Movie-night RAG: recommending what to watch from my saved reels

**Final project submission** · Jean-Paul Wenger · October 2026
Code: [github.com/jpweng/movierag](https://github.com/jpweng/movierag)

### Summary

I built a retrieval-augmented generation (RAG) system that answers movie-night requests in
plain language (*"a war movie under 2 hours"*, *"something like Under the Skin"*) from 91
film and series clips I had saved on Facebook and Instagram, and tops up with new titles from
TMDb and YouTube when my list has nothing suitable. The saved reels become an enriched
catalogue of 85 titles; requests are parsed into hard constraints plus a search text, matched
with hybrid search (embeddings + BM25 + reranking), and answered by a 4-bit Qwen3-4B running on
an 8 GB laptop GPU, with every recommendation citing a retrieved candidate.

**Key results** on the evaluation set (section 5):

| Layer | Metric | Result |
|---|---|---|
| Retrieval (30 questions) | MRR, vector only → hybrid + rerank | 0.918 → **0.983** |
| Constraint parsing (12 requests) | per-field accuracy, LLM + rules | 0.96 |
| Generation (6 requests) | answers citing a non-existent candidate | **0 / 6** |
| Generation (6 requests) | LLM-as-judge faithfulness | 5 / 6 PASS |
| Cost | GPU memory, median latency | 3.9 GB, 16 s per answer |

Reading the answers one by one shows weaknesses the metrics don't capture; they are
discussed in section 6.

## How to run this notebook

This notebook is the single entry point for the project. It rebuilds the catalogue, the
indexes and the models from the raw data, then demonstrates and evaluates the system.
The implementation lives in the `movierag/` package (one module per concept); this notebook
calls it and shows the results.

1. `uv sync --extra dev` in the project folder, then pick the project's environment as the kernel.
2. Copy `.env.example` to `.env` and add a `TMDB_API_KEY` (free). `YOUTUBE_API_KEY` is optional.
3. Run all cells top to bottom. With the models already downloaded and API responses cached,
   it takes about 5 minutes on an 8 GB laptop GPU. The first run downloads ~8 GB of model weights.

No GPU? Set `LLM_BACKEND=ollama` or `openai_like` in `.env`. To check only the plumbing
(mock models, no network), set `MOVIERAG_OFFLINE=true` or run `uv run pytest -q`.

## 1. Problem and approach

Over time I saved 91 short movie clips ("reels") on Facebook and Instagram, each one a film or
series I meant to watch. In practice that list is useless when it's time to pick something:
captions are short, multilingual and full of hashtags, and nothing says how long a film is or
what it's like.

**Goal:** ask in plain language (*"a war movie under 2 hours"*, *"something like Under the
Skin"*) and get recommendations **from my own list first**, topped up with **new** titles
when the list has nothing suitable. Every recommendation must be grounded in real metadata,
not invented by the model.

This is a retrieval-augmented generation (RAG) problem with two twists: requests mix
**fuzzy wishes** (mood, theme) with **hard constraints** (runtime, years, format), and the
corpus is tiny (85 titles), so retrieval quality at the top of the ranking is what matters.

```
OFFLINE (once)                                         ONLINE (per question)
movies-reels.json                                      "something like Devs but a movie, < 2h"
  │ group reels by IMDb id                               │ 1. parse    LLM + regex rules → QueryPlan
  ▼                                                      │ 2. filter   hard constraints in Python
distinct titles ──TMDb──▶ plot, genres, runtime,         │ 3. retrieve vector + BM25 → RRF → rerank
                          cast, keywords, streaming      │ 4. discover TMDb / YouTube, if needed
                ──IMDb──▶ ratings                        │ 5. generate LLM answer citing [L1] / [N2]
  ▼                                                      │ 6. check    every citation must exist
catalogue.json ──▶ Chroma (vectors) + BM25 (keywords) ───┘
```

| Component | Choice | Why |
|---|---|---|
| Enrichment | TMDb API + IMDb ratings dump | IMDb has no free API; TMDb can look titles up by IMDb id |
| Embeddings | `BAAI/bge-small-en-v1.5` (33M params) | strong for its size, runs on CPU |
| Keyword search | BM25 | names and rare words (*Paganini*, *Shaolin*) |
| Fusion | Reciprocal Rank Fusion | merges rankings without calibrating scores |
| Reranker | `cross-encoder/ms-marco-MiniLM-L6-v2` | sharper ordering of the shortlist |
| Vector store | Chroma, persisted | fingerprinted, rebuilt only when the catalogue changes |
| LLM | Qwen3-4B-Instruct-2507, 4-bit NF4 | fits in ~2.5 GB of VRAM; answers without a "thinking" phase |
| Framework | LlamaIndex | swappable LLM backends (HF, Ollama, OpenAI-compatible) |

### Setup

In [2]:
import os, sys, pathlib

# Must be set before transformers is imported: show its warnings and errors.
os.environ["TRANSFORMERS_VERBOSITY"] = "warning"

# Make the project importable even if it wasn't installed (uv sync installs it).
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_colwidth", 70)

from movierag.config import get_config
cfg = get_config()
print(cfg.summary())

bookmarks     : /home/jp/llms/rag/movies-reels.json
state dir     : /home/jp/.movierag
LLM           : hf:Qwen/Qwen3-4B-Instruct-2507 (4-bit)
embeddings    : BAAI/bge-small-en-v1.5
reranker      : cross-encoder/ms-marco-MiniLM-L6-v2
TMDb key      : yes
YouTube key   : yes
watch region  : CH
offline mode  : False


## 2. Data: from saved reels to a catalogue

### 2.1 The raw bookmarks

`movies-reels.json` has one record per saved reel: platform, caption, and a best guess of
which title it shows (with an IMDb link and a confidence status).

In [3]:
from movierag.catalogue import load_bookmarks, group_bookmarks

bookmarks = load_bookmarks(cfg.bookmarks_path)
reels = pd.DataFrame([b.model_dump() for b in bookmarks])
print(len(reels), "saved reels\n")
for col in ["platform", "match_status"]:
    print(reels[col].value_counts().to_string(), "\n")
reels[["id", "platform", "title", "year", "kind", "match_status", "caption"]].head(8)

91 saved reels

platform
Facebook     85
Instagram     6 

match_status
Confirmed           80
Candidate            8
Unresolved           2
Confirmed (TMDb)     1 



,id,platform,title,year,kind,match_status,caption
0,FB001,Facebook,The Looking Planet,2014.0,Short,Confirmed,The Looking Planet
1,FB002,Facebook,Pipeline,2021.0,Film,Confirmed,Ce film
2,FB003,Facebook,Devs,2020.0,TV mini series,Confirmed,Questions whether free will remains meaningful when an AI can pred...
3,FB004,Facebook,A Very Long Engagement (Un long dimanche de fiançailles),2004.0,Film,Confirmed,Des soldats essaient de rentré chez eux #resumedefilm #drama #fran...
4,FB005,Facebook,Siegfried,2005.0,Film,Confirmed,Le Jeune Homme Sauvage Qui Pouvait Parler aux Animaux..#film #reel...
5,FB006,Facebook,Oruzhie (The Weapon),2012.0,TV movie,Candidate,🎥🎬🍿
6,FB007,Facebook,The Count of Monte Cristo,2024.0,Film,Candidate,Fimz12's Video
7,FB008,Facebook,Mortal Kombat II,2026.0,Film,Confirmed,🔥 One of the BEST scenes in Mortal Kombat Johnny Cage vs. Baraka W...


The captions alone would make poor search documents: they mix French, English, Thai and
Chinese, are often just hashtags, and say nothing about runtime or genre. Several reels can
also point to the same title, or to one episode of a series.

### 2.2 One record per title

Reels are grouped by IMDb id. Unresolved reels (no identifiable title) are dropped.

In [4]:
groups, skipped = group_bookmarks(bookmarks)
print(f"{len(bookmarks)} reels -> {len(groups)} distinct titles, {len(skipped)} unresolved (dropped)")
for key, group in groups.items():
    if len(group) > 1:
        print(f"  saved more than once: {group[0].title} ({key})")

91 reels -> 87 distinct titles, 2 unresolved (dropped)
  saved more than once: Kalki 2898 AD (imdb:tt12735488)
  saved more than once: Fortress (imdb:tt0106950)


### 2.3 Enrichment with TMDb and IMDb

For each title: TMDb `/find` by IMDb id, then `/movie/{id}` or `/tv/{id}` for plot, runtime,
genres, keywords, credits and streaming providers in Switzerland (`WATCH_REGION=CH`).
IMDb ratings come from IMDb's public non-commercial dataset. Episodes are merged into their
series, so the number of titles can drop further. API responses are cached on disk, so this
cell is fast and reproducible after the first run.

In [5]:
from movierag.tmdb import TMDbClient
from movierag.catalogue import build_catalogue, save_catalogue

tmdb = TMDbClient.from_config(cfg)
titles = build_catalogue(bookmarks, tmdb, ratings_cache_dir=cfg.cache_dir)
save_catalogue(titles, cfg.catalogue_path)

91 reels -> 87 distinct titles (2 unresolved, skipped)
  enriched 10/87
  enriched 20/87
  enriched 30/87
  enriched 40/87
  enriched 50/87
  enriched 60/87
  enriched 70/87
  enriched 80/87
IMDb ratings attached to 84 titles
catalogue: 85 titles, 85 enriched from TMDb, TMDb network calls: 0


In [6]:
cat = pd.DataFrame([{
    "title": t.title, "kind": t.kind, "year": t.year, "min": t.runtime_min,
    "genres": ", ".join(t.genres[:3]), "IMDb": t.imdb_rating,
    "streaming (CH)": ", ".join(t.providers[:2]), "reels": len(t.bookmarks), "confidence": t.confidence,
} for t in titles])

coverage = pd.Series({
    "plot": sum(bool(t.overview.strip()) for t in titles),
    "runtime": sum(t.runtime_min is not None for t in titles),
    "genres": sum(bool(t.genres) for t in titles),
    "keywords": sum(bool(t.keywords) for t in titles),
    "IMDb rating": sum(t.imdb_rating is not None for t in titles),
    "streaming in CH": sum(bool(t.providers) for t in titles),
}, name="titles with field")
print(f"{len(titles)} titles:", cat["kind"].value_counts().to_dict())
print(f"\nField coverage (out of {len(titles)}):\n" + coverage.to_string())

85 titles: {'film': 63, 'series': 19, 'short': 3}

Field coverage (out of 85):
plot               85
runtime            85
genres             85
keywords           74
IMDb rating        84
streaming in CH    38


In [7]:
cat.sort_values("IMDb", ascending=False).head(10)

,title,kind,year,min,genres,IMDb,streaming (CH),reels,confidence
36,Doctor Who,series,2005,60,"Action, Adventure, Drama",8.5,,3,confirmed
82,Requiem for a Dream,film,2000,102,"Crime, Drama",8.3,Canal+,1,confirmed
72,Bloodhounds,series,2023,61,"Drama, Action, Adventure",8.2,Netflix,1,candidate
79,Something the Lord Made,film,2004,110,Drama,8.0,,1,confirmed
55,WandaVision,series,2021,51,"Science Fiction, Fantasy, Mystery",7.9,Disney Plus,1,confirmed
54,Life of Pi,film,2012,127,"Adventure, Drama",7.9,Disney Plus,1,confirmed
52,Da Vinci's Demons,series,2013,60,Drama,7.9,,1,confirmed
80,Black Narcissus,film,1947,100,Drama,7.7,"ARD Plus Amazon channel , Artiflix",1,confirmed
3,A Very Long Engagement,film,2004,133,"Drama, Romance, War",7.6,,1,confirmed
2,Devs,series,2020,52,"Science Fiction, Fantasy, Drama",7.6,Disney Plus,1,confirmed


### 2.4 The document that gets indexed

Choosing what goes into each title's text is one of the most important decisions in a RAG
system. Each title becomes one document (texts are short, so no chunking): title, kind,
genres, tagline, plot, keywords, director, cast, and the caption of the reel I saved.

In [8]:
devs = next(t for t in titles if t.imdb_id == "tt8134186")
print(devs.embedding_text())

Devs (2020) - series
Genres: Science Fiction, Fantasy, Drama, Mystery
Tagline: Reality is stranger than dreams.
Plot: Young computer engineer Lily Chan investigates the secretive development division of her employer, a cutting-edge tech company based in San Francisco, which she believes is behind the murder of her boyfriend.
Keywords: simulation, silicon valley, mysterious, complex, anxious, determinism, intimate, baffled, distressing
By: Alex Garland
Cast: Sonoya Mizuno, Nick Offerman, Jin Ha, Cailee Spaeny, Stephen McKinley Henderson
Saved reel said: Questions whether free will remains meaningful when an AI can predict a person's choices almost perfectly; tags mention quantum computing and science fiction.


The catalogue now holds two kinds of information, and each needs a different tool:

| Kind | Example | Handled by |
|---|---|---|
| structured | runtime = 104 min, year = 2014 | **filters** |
| unstructured | "a lonely alien in Scotland" | **search** |

## 3. Retrieval

### 3.1 Loading the models

One call wires everything together: catalogue, embedding model, Chroma + BM25 indexes,
reranker, TMDb/YouTube clients and the 4-bit LLM.

In [9]:
from movierag.app import load_components
from movierag.models import gpu_report

c = load_components()
index, reranker, llm, recommender = c.index, c.reranker, c.llm, c.recommender
print(len(index.nodes), "documents indexed")
print(gpu_report())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

85 documents indexed
NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU: 3.9 GB used / 8.0 GB


### 3.2 Embeddings capture meaning, not words

A sentence-embedding model maps a text to a 384-dimensional vector. Texts with similar
meaning point in similar directions, even with no words in common.

In [10]:
import numpy as np

sentences = [
    "A submarine captain defies the US Navy",
    "Tense standoff between a nuclear sub and an aircraft carrier",
    "A teenage girl learns to bake croissants in Paris",
]
vecs = np.array([c.embed_model.get_text_embedding(s) for s in sentences])
pd.DataFrame(vecs @ vecs.T, index=[s[:40] for s in sentences], columns=["sub", "carrier", "croissants"]).round(2)

,sub,carrier,croissants
A submarine captain defies the US Navy,1.00,0.64,0.29
Tense standoff between a nuclear sub and,0.64,1.00,0.28
A teenage girl learns to bake croissants,0.29,0.28,1.00


### 3.3 Vector search vs BM25 vs hybrid

The two retrievers fail in opposite ways: vector search handles paraphrase but can miss exact
names; BM25 nails rare words but knows no synonyms. **Hybrid** search runs both and merges
the rankings with Reciprocal Rank Fusion:

$$\text{RRF}(d) = \sum_{\text{retrievers}} \frac{1}{k + \text{rank}(d)}, \qquad k = 60$$

Only ranks are used, so a cosine similarity (0–1) and a BM25 score (0–30+) can be combined
without calibration.

In [11]:
def top(hits, n=3):
    return " · ".join(t.title for t, _ in index.titles_for(hits)[:n]) or "(nothing)"

queries = [
    "Paganini",                                          # rare name: BM25 territory
    "a man stranded on a lifeboat with a wild animal",   # paraphrase: vector territory
    "eerie arthouse film about an alien seductress",     # Under the Skin
]
for q in queries:
    print(f"=== {q}")
    for method in ["vector", "bm25", "hybrid"]:
        print(f"  {method:7} {top(index.search(q, method, top_k=3))}")
    print()

=== Paganini
  vector  The Devil's Violinist · Black Narcissus · The Last Witch Hunter
  bm25    The Devil's Violinist
  hybrid  The Devil's Violinist · Black Narcissus · The Last Witch Hunter

=== a man stranded on a lifeboat with a wild animal
  vector  Life of Pi · The Abyss · Mickey 17
  bm25    Life of Pi · Masters of the Universe · Memories
  hybrid  Life of Pi · The Protector · Noah

=== eerie arthouse film about an alien seductress
  vector  The Abyss · Hell Hole · Alien Invasion: S.U.M.1
  bm25    Under the Skin · A Quiet Place Part II · Alien Invasion: S.U.M.1
  hybrid  A Quiet Place Part II · Alien Invasion: S.U.M.1 · Under the Skin



### 3.4 Reranking the shortlist

Embeddings are a *bi-encoder*: query and documents are embedded separately, which is fast but
means the model never reads them together. A *cross-encoder* reads each (query, document)
pair in one pass. It's sharper and slower, so it only reorders the top ~15 candidates.

In [12]:
q = "a slow, unsettling science fiction film with almost no dialogue"
print("hybrid:         ", top(index.search(q, "hybrid", top_k=5), 5))
print("hybrid + rerank:", top(index.search(q, "hybrid", top_k=5, reranker=reranker), 5))

hybrid:          Devs · Under the Skin · Automata · Alien Invasion: S.U.M.1 · Calculator
hybrid + rerank: Under the Skin · A Quiet Place Part II · The Abyss · Hell Hole · Vivarium


### 3.5 Hard constraints are filters, not search

"Under 90 minutes" is a number, not a vibe. Constraints are applied as a set of allowed ids
before search (a Chroma metadata `IN` filter, and a BM25 index over the allowed documents).

In [13]:
from movierag.schemas import QueryPlan
from movierag.recommender import filter_titles

plan = QueryPlan(search_text="war and soldiers", kinds=["series"])
allowed, relaxed = filter_titles(titles, plan)
print(len(allowed), "series in the list ->", top(index.search(plan.search_text, "hybrid", top_k=5, allowed_ids=allowed), 5))

19 series in the list -> See · Raised by Wolves · Tiger and Crane · Avatar: The Last Airbender · Doctor Who


## 4. Generation: the full RAG loop

### 4.1 Self-querying: from a sentence to a structured plan

The LLM fills a Pydantic schema (`QueryPlan`) through LlamaIndex's `structured_predict`.
Small models make mistakes, so a regex parser runs alongside: its explicit numbers (runtime,
years) override the LLM's, and if the LLM returns invalid JSON the rules are used alone.
`[llm]`, `[llm+rules]` or `[rules]` shows which happened.

In [14]:
from movierag.query_parser import parse_query

for q in [
    "a war movie, under 2 hours, with tanks",
    "something like Under the Skin but a series",
    "a documentary about the ocean, less than 90 minutes",
    "an 80s or older kung fu movie",
]:
    plan, how = parse_query(q, llm)
    print(f"{q}\n   [{how}] search='{plan.search_text}' | {plan.constraints_text()}\n")

a war movie, under 2 hours, with tanks
   [llm] search='war, tanks, military conflict' | format: film, genres: War, <= 120 min

something like Under the Skin but a series
   [llm] search='body horror, alienation, identity, isolation' | format: series, genres: Horror/Drama, like 'Under the Skin'

a documentary about the ocean, less than 90 minutes
   [llm] search='ocean life, marine ecosystems, conservation efforts' | format: documentary, genres: Documentary, <= 90 min

an 80s or older kung fu movie
   [llm+rules] search='kung fu, martial arts, action, revenge, destiny' | genres: Action/Adventure, >= 1980, <= 1989



### 4.2 End to end

`recommend()` runs parse → filter → hybrid retrieve → rerank → (discover) → generate → check.
Candidates from my list are labelled `[L1]…`, new ones `[N1]…`; the answer must cite them,
and any label that doesn't exist is flagged.

In [15]:
r = recommender.recommend("a war movie, under 2 hours, with tanks")
r.show()

Q: a war movie, under 2 hours, with tanks
plan (llm): search='war, tanks, military conflict' | format: film, genres: War, <= 120 min

From your list:
  [L1] The Tank (2025) film, 117 min  score=3.355
  [L2] White Tiger (2012) film, 105 min  score=0.084
  [L3] Prisoner of War (2025) film, 113 min  score=-4.511
  [L4] The Silent Service (2023) film, 113 min  score=-6.913
  [L5] See You Up There (2017) film, 113 min  score=-7.707

- [L1] The Tank (2025, 117 min) – A war film featuring a tank crew in WWII, with direct focus on tanks and a mission set in hostile terrain. Fits the war genre and runtime (under 120 min), with explicit tank action and wartime setting.  
- [L2] White Tiger (2012, 105 min) – Set in 1945 during WWII, centers around a Red Army soldier confronting a ghostly-white German Tiger tank. Includes tanks as a key element and fits the war genre and under-120-minute runtime.  
- [L3] Prisoner of War (2025, 113 min) – A WWII war film involving captivity and combat, though it f

The LLM never "knows" my films. This is the whole prompt it sees: the question, the parsed
constraints and the retrieved candidate cards. When an answer is wrong, the first question
is always whether the right film was in this prompt.

In [16]:
from movierag.recommender import build_prompt
print(build_prompt(r))

You are a friendly movie-night assistant. The user asked:
"a war movie, under 2 hours, with tanks"

Their constraints: format: film, genres: War, <= 120 min

CANDIDATES FROM THE USER'S SAVED LIST:
[L1] The Tank (film, 2025, 117 min, War/Action/Drama, IMDb 6.5, streaming: Amazon Prime Video)
A German Tiger tank crew is sent on a dangerous mission to rescue the missing officer Paul von Hardenburg from a top-secret bunker behind enemy lines. As they make their way through the lethal no-man's land, they must confront not only the enemy, but also their own fears and inner demons. Fueled by the Wehrmacht's methamphetamine, their mission increasingly becomes a journey into the heart of darkness.

[L2] White Tiger (film, 2012, 105 min, Action/Adventure/Fantasy, IMDb 6.1)
WWII, 1945. After barely surviving a battle with a mysterious, ghostly-white German Tiger tank, Red Army Sergeant Ivan Naydenov becomes obsessed with its destruction.

[L3] Prisoner of War (film, 2025, 113 min, Action/War/Thri

### 4.3 When the list isn't enough: discovery

I saved no documentaries. When the list can't satisfy a request (or the user asks for new
titles), the recommender queries TMDb: item-to-item recommendations for a reference title,
recommendations seeded by the best matches from my list, and `/discover` with the plan's
genres, runtime, years and topic keywords. New candidates are ranked with the same embedding
model and reranker, and titles already in my list are excluded.

In [17]:
recommender.recommend("a documentary about the ocean, less than 90 minutes").show()

Q: a documentary about the ocean, less than 90 minutes
plan (llm): search='ocean life, marine ecosystems, conservation efforts' | format: documentary, genres: Documentary, <= 90 min
  relaxed: no saved title matched genres ['Documentary']; ignored the genre filter

From your list:
New:
  [N1] Ocean with David Attenborough (2025) documentary, 85 min  score=-2.689  https://www.themoviedb.org/movie/1448497
  [N2] The Seafarers (1953) documentary, 29 min  score=-10.088  https://www.themoviedb.org/movie/45314
  [N3] To the Sea (2010) documentary, 73 min  score=-10.386  https://www.themoviedb.org/movie/48414

- [N1] Ocean with David Attenborough (2025, 85 min)  
A visually stunning and scientifically rich documentary that explores the ocean’s vital role in planetary health. With breathtaking cinematography and expert narration, it offers an engaging, educational look at marine ecosystems without exceeding the 90-minute limit.

- [N3] To the Sea (2010, 73 min)  
A heartfelt family documentary

For *"like X"* requests where X is in my list, X's own plot is added to the search query, and
X itself is excluded from the results.

In [18]:
recommender.recommend("something like Under the Skin").show()

Q: something like Under the Skin
plan (llm): search='alien existence, isolation, body horror, existential dread' | genres: Science Fiction/Horror, like 'Under the Skin'

From your list:
  [L1] A Quiet Place Part II (2021) film, 97 min  score=-7.277
  [L2] Hell Hole (2024) film, 92 min  score=-7.826
  [L3] Alien Invasion: S.U.M.1 (2017) film, 95 min  score=-8.787
  [L4] The Abyss (1989) film, 140 min  score=-9.161
  [L5] Vivarium (2019) film, 99 min  score=-9.781

- [L1] A Quiet Place Part II (2021) – Continues the sci-fi horror theme of silent predators and survival under extreme threat, with a strong atmospheric tension and alien-like dangers that echo the body horror and isolation of *Under the Skin*.  
- [L5] Vivarium (2019) – Features a surreal, isolated setting and a disturbing sense of otherness, with a trapped couple forced to survive in a mysterious, non-human environment—aligning with the eerie, body-horror tone of *Under the Skin*.  

No other candidates strongly match the bl

## 5. Evaluation

An answer that *sounds* good says little. The system is measured in three layers, cheapest
first, on the evaluation set in `eval/questions.json`:

1. **Retrieval**: are the right titles in the top 5?
2. **Parsing**: are the constraints extracted correctly?
3. **Generation**: is the answer grounded in what was retrieved?

In [19]:
from movierag.evaluate import load_eval_set

data = load_eval_set(cfg.bookmarks_path.parent / "eval" / "questions.json")
print(len(data["retrieval"]), "retrieval questions,", len(data["parsing"]), "parsing cases,",
      len(data["end_to_end"]), "end-to-end questions")

30 retrieval questions, 12 parsing cases, 6 end-to-end questions


### 5.1 Retrieval ablation

Same 30 questions, four retrieval setups, each with a labelled set of relevant titles.

- **hit@5**: is at least one relevant title in the top 5?
- **recall@5**: what share of the relevant titles is in the top 5?
- **MRR** (mean reciprocal rank): 1 if the first result is relevant, ½ if the second is, and so on.

In [20]:
from movierag.evaluate import evaluate_retrieval

def searcher(method, reranker=None):
    def search(q):
        hits = index.search(q, method=method, top_k=5, reranker=reranker)
        return [index.by_id[h.node.node_id].imdb_id for h in hits]
    return search

setups = {
    "vector": searcher("vector"),
    "bm25": searcher("bm25"),
    "hybrid": searcher("hybrid"),
    "hybrid + rerank": searcher("hybrid", reranker),
}
results, details = {}, {}
for name, fn in setups.items():
    results[name], details[name] = evaluate_retrieval(data["retrieval"], fn, k=5)

retrieval_table = pd.DataFrame(results).T.round(3)
retrieval_table

,hit@5,recall@5,mrr
vector,1.0,0.972,0.918
bm25,1.0,0.961,0.922
hybrid,1.0,0.989,0.967
hybrid + rerank,1.0,0.989,0.983


Averages hide the interesting part, so here is every question where the best setup did not
rank a relevant title first:

In [21]:
by_imdb = {t.imdb_id: t.title for t in titles}
for name in setups:
    misses = [(row, q) for row, q in zip(details[name], data["retrieval"]) if row["mrr"] < 1]
    print(f"{name}: {len(misses)} of {len(data['retrieval'])} questions without a relevant title at rank 1")

print()
for row, q in zip(details["hybrid + rerank"], data["retrieval"]):
    if row["mrr"] < 1:
        print(f"MRR {row['mrr']:.2f} | {row['question']}")
        print("    wanted:", [by_imdb.get(i, i) for i in q["relevant_imdb"]])
        print("    got   :", [by_imdb.get(i, i) for i in row["top"]])

vector: 4 of 30 questions without a relevant title at rank 1
bm25: 4 of 30 questions without a relevant title at rank 1
hybrid: 2 of 30 questions without a relevant title at rank 1
hybrid + rerank: 1 of 30 questions without a relevant title at rank 1

MRR 0.50 | a clone sent on dangerous missions who keeps dying and being reprinted
    wanted: ['Mickey 17']
    got   : ['The Tank', 'Mickey 17', 'Die Another Day', 'Flight of Fury', 'Alien Invasion: S.U.M.1']


### 5.2 Query parsing: rules alone vs LLM + rules

Per-field accuracy on 12 requests with hand-labelled expected constraints.

In [22]:
from movierag.evaluate import plan_field_matches

rows = []
for item in data["parsing"]:
    for mode, model in (("rules", None), ("llm+rules", llm)):
        plan, how = parse_query(item["question"], model)
        for field, ok in plan_field_matches(plan, item["expected"]).items():
            rows.append({"mode": mode, "field": field, "ok": ok, "question": item["question"], "parsed_by": how})

parse_df = pd.DataFrame(rows)
parse_table = parse_df.pivot_table(index="field", columns="mode", values="ok", aggfunc="mean").round(2)
parse_table.loc["overall"] = parse_df.groupby("mode")["ok"].mean().round(2)
parse_table

LLM query parsing failed (1 validation error for QueryPlan
kinds
  Input should be a valid array [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type); using rules only


mode,llm+rules,rules
field,,
genres,1.00,1.0
kinds,1.00,1.0
max_runtime_min,1.00,1.0
max_year,1.00,1.0
min_year,0.67,1.0
similar_to,1.00,1.0
wants_new,1.00,1.0
overall,0.96,1.0


In [23]:
print("LLM parse fell back to rules only on:",
      sorted(set(parse_df.query("mode == 'llm+rules' and parsed_by == 'rules'")["question"])) or "none")
parse_df[~parse_df["ok"]].sort_values(["mode", "question"])

LLM parse fell back to rules only on: ['a tense submarine thriller from the 90s']


,mode,field,ok,question,parsed_by
20,llm+rules,min_year,False,an 80s or older kung fu movie,llm+rules


### 5.3 Generation: grounded or not?

Two checks on each end-to-end answer:

- **Citations** (free, deterministic): every `[L…]`/`[N…]` in the answer must exist among the candidates.
- **LLM-as-judge**: a second LLM call grades whether every recommended title and claim is
  supported by the candidate cards. Here the judge is the same 4B model being graded, which is
  a known weakness (see the discussion).

In [24]:
from movierag.evaluate import judge_faithfulness
from movierag.recommender import context_text

gen_rows = []
for q in data["end_to_end"]:
    r = recommender.recommend(q)
    mine, new = context_text(r)
    passed, verdict = judge_faithfulness(llm, r.answer, mine + "\n\n" + new)
    gen_rows.append({
        "question": q, "parsed_by": r.parsed_by, "from list": len(r.my_list), "new": len(r.new),
        "cited": len(r.cited), "bad citations": len(r.bad_citations),
        "judge": "PASS" if passed else "FAIL", "seconds": round(r.timings["total"], 1),
    })
    print(f"--- {q}\n{r.answer}\n>> judge: {verdict}\n")

LLM query parsing failed (1 validation error for QueryPlan
kinds
  Input should be a valid array [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type); using rules only


--- I have 90 minutes tonight, something gripping
- [L1] Bloodhounds (2023) – A gripping 61-minute drama-action series about two young boxers uniting to fight ruthless foes and protect their loved ones; high-stakes, action-driven, and emotionally intense, perfect for a thrilling 90-minute movie-night.  
- [L3] The Weapon (2012) – An 85-minute drama centered on military weapons testing and tense moral dilemmas; features high tension, real-world stakes, and a strong narrative arc involving danger and secrecy.  
- [L4] Tomb Adventurer (2022) – A full 90-minute action-fantasy film with mythical beasts, high-risk adventures, and a clear quest to escape a deadly labyrinth; packed with suspense and excitement, ideal for a gripping evening.  

*Note: [L2] Devs is 52 minutes per episode and not a single film, making it unsuitable for a 90-minute continuous watch. [L5] Subdue the Devil is 82 minutes but has lower ratings and less emphasis on sustained gripping tension.*
>> judge: PASS  
Each rec

In [25]:
gen_df = pd.DataFrame(gen_rows)
gen_df

,question,parsed_by,from list,new,cited,bad citations,judge,seconds
0,"I have 90 minutes tonight, something gripping",rules,5,0,5,0,PASS,22.1
1,"a war movie, but not too long",llm,5,0,3,0,PASS,21.1
2,a documentary about the ocean,llm,0,3,3,0,PASS,18.0
3,something like Under the Skin,llm,5,0,2,0,PASS,18.6
4,new sci-fi series I haven't saved,llm,5,3,3,0,PASS,15.8
5,a youtube video essay about kung fu cinema under 30 minutes,llm,0,3,3,0,FAIL,17.5


In [26]:
print(f"answers with invalid citations : {(gen_df['bad citations'] > 0).sum()} / {len(gen_df)}")
print(f"judge PASS                     : {(gen_df['judge'] == 'PASS').sum()} / {len(gen_df)}")
print(f"median latency                 : {gen_df['seconds'].median():.1f} s per question")
print(gpu_report())

answers with invalid citations : 0 / 6
judge PASS                     : 5 / 6
median latency                 : 18.3 s per question
NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU: 4.2 GB used / 8.0 GB


## 6. Discussion

### 6.1 Retrieval

Hybrid search beats either retriever alone, and the reranker fixes most of what is left:
the number of questions without a relevant title at rank 1 drops from 4 (vector or BM25) to
2 (hybrid) to 1 (hybrid + rerank). Section 3.3 shows why: BM25 alone finds *Under the Skin*
from "alien seductress", vector search alone finds *Life of Pi* from a paraphrase, and fusion
keeps both strengths.

hit@5 is 1.0 for every setup, so this metric has hit its ceiling: the evaluation set is too
easy to separate the setups at the top-5 level. The questions were written knowing the
catalogue, most have a single relevant title and they describe it quite specifically. MRR is
the more informative number here. The one remaining miss (*Mickey 17*, ranked second behind
*The Tank* for "a clone sent on dangerous missions") is a plausible near-miss rather than a
broken result.

### 6.2 Query parsing

The rules alone score 1.00 and LLM + rules 0.96, but this comparison flatters the rules: they
were written with requests like these in view, and the score only covers the structured fields.
The LLM's contribution is what isn't scored: turning vague wording into a search text
("war, tanks, military conflict") and genres.

The single LLM + rules error points to a design gap. For "an 80s or older kung fu movie" the
rules correctly leave `min_year` empty, but the merge step only overrides the LLM when the rules
*found* a value, so the LLM's `min_year = 1980` survives. The rules need a way to say
"explicitly unbounded" as opposed to "not found".

The fallback design did its job: on two requests the 4B model returned `kinds: null`, which
fails schema validation, and the system answered with the rule-based plan instead of crashing.

### 6.3 Generation

The deterministic citation check found no invented candidates in 6 answers, and the prompt
(section 4.2) keeps the model to the retrieved cards. But reading the answers shows that
*grounded* is not the same as *good*:

- **"new sci-fi series I haven't saved"**: two of the three picks come from the saved list.
  The prompt says to recommend saved titles first, which contradicts `wants_new`. The judge
  passed it because it checks faithfulness, not relevance.
- **"a youtube video essay about kung fu cinema"**: discovery returned martial-arts action and
  training clips, not video essays, and the answer presents them as good fits. The judge failed
  it, but the reason it gives cites details that are not in the answer: the right verdict for
  the wrong reason.
- **"I have 90 minutes tonight"**: the top pick is a series with 61-minute episodes, because
  the runtime filter compares the episode length.

A judge that is the same 4B model it grades is a weak instrument: it passes answers that ignore
part of the request and can invent reasons for its verdicts. Its PASS rate should be read as
an upper bound.

### 6.4 Cost

Everything runs locally in 3.9 GB of GPU memory. Retrieval takes under 0.1 s; a full answer
takes a median of 16 s, almost all of it LLM generation (parsing ~3–5 s, the answer ~9–18 s).
Timings vary by a second or two between runs.

## 7. Limitations

- **Small evaluation set:** 30 retrieval questions, 12 parsing cases and 6 end-to-end
  requests. The answer keys were drafted with an AI assistant from the catalogue, not labelled
  independently, so the retrieval numbers are likely optimistic.
- **Self-grading:** the LLM judge is the model being evaluated.
- **Reproducibility:** generation is greedy and API responses are cached, so a re-run on the
  same machine gives the same results. With a fresh cache, TMDb and YouTube may return different
  discoveries.
- **Language:** many saved captions are in French, Thai or Chinese, while the embedding model is
  English-only; the captions only contribute one line of each document.

## 8. Future work

| Change | Expected effect |
|---|---|
| Multilingual embeddings (`BAAI/bge-m3`) | use the non-English captions |
| Stronger reranker (`BAAI/bge-reranker-v2-m3`) | MRR on harder questions |
| Prompt: new suggestions first when `wants_new` is set | fixes the "haven't saved" failure |
| Filter series on total length, or flag them for time-boxed requests | fixes the "90 minutes" failure |
| Let rules mark a bound as explicitly open | fixes the "80s or older" parsing error |
| A stronger judge, e.g. the 30B Qwen in `deploy/modal_vllm.py`, grading relevance too | trustworthy generation scores |
| A harder evaluation set: vague requests, several relevant titles, independent labels | separates the retrieval setups at hit@5 |

## 9. Data sources and references

- This product uses the TMDb API but is not endorsed or certified by TMDb.
- IMDb ratings: [IMDb non-commercial datasets](https://developer.imdb.com/non-commercial-datasets/), personal and non-commercial use.
- YouTube Data API v3.
- Cormack, Clarke & Büttcher (2009), *Reciprocal Rank Fusion outperforms Condorcet and individual rank learning methods*, SIGIR.
- Models: `BAAI/bge-small-en-v1.5`, `cross-encoder/ms-marco-MiniLM-L6-v2`, `Qwen/Qwen3-4B-Instruct-2507`. Framework: LlamaIndex, Chroma.